In [ ]:
# Configuración de entorno
ENTORNO = "local" # Opciones: "local" o "colab"
ALMACENAMIENTO = "local" # Opciones: "local" o "drive" (solo aplica si ENTORNO = "colab")

# Rutas base según la configuración
RUTA_DRIVE = '/content/drive/MyDrive/TFM'
RUTA_LOCAL = '..' if os.path.basename(os.getcwd()) == 'MACE' else '.' # Ajuste automático de directorio raíz

import sys

if ENTORNO == "colab":
    print("Ejecutando en entorno Google Colab...")
    dispositivo = "cuda"
    if ALMACENAMIENTO == "drive":
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            os.chdir(RUTA_DRIVE)
            print(f"Montado Drive y trabajando en {RUTA_DRIVE}")
        except ImportError:
            print("No se pudo montar drive. ¿Seguro que estás en Colab?")
    else:
        print("Usando almacenamiento local de la sesión de Colab.")
    
    # Instalamos dependencias si estamos en Colab
    print("Comprobando e instalando dependencias (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
    get_ipython().system('pip install -q cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12')
elif ENTORNO == "local":
    os.chdir(RUTA_LOCAL)
    dispositivo = "cpu" # Forzamos CPU para local según especificaciones
    print(f"Ejecutando en entorno local usando {dispositivo.upper()}. Directorio actual: {os.getcwd()}")
    if 'google.colab' in sys.modules:
        print("ADVERTENCIA: Parece que estás en Colab pero has seleccionado entorno 'local'.")
    
    print("Comprobando e instalando dependencias locales (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
else:
    print("Entorno no reconocido. Usa 'colab' o 'local'.")
    dispositivo = "cpu"


# Load libraries and set device up

In [ ]:
import importlib
import libraries.graph as clg

try:
    cld = importlib.reload(cld)
except NameError:
    pass

try:
    clg = importlib.reload(clg)
except NameError:
    pass

In [ ]:
import numpy as np
import torch.nn as nn
import os
import torch
from pathlib import Path
import importlib

import libraries.model as clm
import libraries.dataset as cld
import libraries.graph as clg

clm = importlib.reload(clm)
cld = importlib.reload(cld)
clg = importlib.reload(clg)

from torch_geometric.loader import DataLoader

# Checking if pytorch can run in GPU, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import os
print(os.getcwd())

In [ ]:
workspace_root = Path.cwd()
while not (workspace_root / 'input' / 'candidates').exists() and workspace_root.parent != workspace_root:
    workspace_root = workspace_root.parent

print(f"Using repository root: {workspace_root}")

data_root = workspace_root / 'input' / 'candidates'

In [ ]:
from datetime import datetime

# Create timestamped folder for this model run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
timestamped_folder = f'{target_folder}/results_{timestamp}'
os.makedirs(timestamped_folder, exist_ok=True)

print(f'Model results will be saved to: {timestamped_folder}')

# Define parameters

In [ ]:
n_epochs =      50
batch_size =    128
learning_rate = 1e-4
dropout =       0.2
patience =      n_epochs//4
delta =         0.1
train_ratio =   0.8
val_ratio =     0.1
test_ratio =    0.1
max_samples =   None
targets = "3d"       # Opciones: "3d" o "multitarget"
model_type = "GCNN"  # Opciones: "GCNN", "DGNN"
optimizer = "Adam"   # Opciones: "Adam", "AdamW", "RAdam", "SGD", "RMSprop"
loss_function = "MSELoss"  # Opciones: "MSELoss", "L1Loss", "SmoothL1Loss"

# Derivar targets, carpeta y n_outputs a partir de la selección
if targets == "3d":
    mis_targets = ['E_3D']
    folder = '3d-energy'
    # PyTorch Geometric creará un tensor 'y' de tamaño [1]
elif targets == "multitarget":
    mis_targets = ['E_1D', 'E_2D', 'E_3D']
    folder = 'all-energies'
    # PyTorch Geometric creará un tensor 'y' de tamaño [3]
else:
    raise ValueError(f"targets debe ser '3d' o 'multitarget', se recibió: {targets}")

n_outputs = len(mis_targets)
target_folder = workspace_root / f'models/{folder}'

model_parameters = {
    'n_epochs':      n_epochs,
    'batch_size':    batch_size,
    'learning_rate': learning_rate,
    'dropout':       dropout,
    'patience':      patience,
    'delta':         delta,
    'train_ratio':   train_ratio,
    'val_ratio':     val_ratio,
    'test_ratio':    test_ratio,
    'max_samples':   max_samples,
    'model_type':    model_type,
    'optimizer':     optimizer,
    'loss_function': loss_function,
    'targets':       mis_targets,
    'folder':        folder,
}

files_names = {
    # Reusable files (common for all models)
    'dataset':              f'{target_folder}/dataset.pt',
    'std_parameters':       f'{target_folder}/standardized_parameters.json',
    'dataset_parameters':   f'{target_folder}/dataset_parameters.json',
    
    # Model-specific files (unique per model run)
    'train_dataset_std':    f'{timestamped_folder}/train_dataset_std.pt',
    'val_dataset_std':      f'{timestamped_folder}/val_dataset_std.pt',
    'test_dataset_std':     f'{timestamped_folder}/test_dataset_std.pt',
    'model':                f'{timestamped_folder}/model.pt',
    'model_parameters':     f'{timestamped_folder}/model_parameters.json',
    'model_type':           model_type
}
cld.save_json(files_names, f'{timestamped_folder}/files_names.json')

cld.save_json(model_parameters, files_names['model_parameters'])

print(f"Modo: {targets} → targets={mis_targets}, carpeta={folder}")

# Generate or load graph database for training

In [ ]:
# Try loading the training datasets directly, else generate them
# Generate data
cld.generate_dataset(str(data_root),
                        targets=mis_targets,
                        data_folder=str(target_folder),
                        max_samples=max_samples)

In [ ]:
# Mostrar un ejemplo
data = train_dataset[0]
print(data)

# Detalles útiles
print("label:", data.label)
print("y:", data.y)
print("num_nodes:", data.num_nodes)
print("x.shape:", data.x.shape)
print("edge_index.shape:", data.edge_index.shape)
print("edge_attr.shape:", data.edge_attr.shape)

# Ver los primeros nodos y aristas
print("x[:5]:\n", data.x[:10])
print("edge_index[:, :20]:\n", data.edge_index[:, :20])
print("edge_attr[:20]:\n", data.edge_attr[:20])

In [ ]:
# Load the raw dataset, with corresponding labels, and standardize it
dataset = torch.load(files_names['dataset'], weights_only=False)

# Split datasets
train_dataset, val_dataset, test_dataset = cld.split_dataset(train_ratio, val_ratio, test_ratio, dataset)
del dataset  # Free up CUDA memory

# Standardize train dataset
train_dataset_std, standardized_parameters = cld.standardize_dataset(train_dataset, transformation='inverse-quadratic')

# Standardize test and validation datasets with train parameters
val_dataset_std  = cld.standardize_dataset_from_keys(val_dataset,  standardized_parameters)
test_dataset_std = cld.standardize_dataset_from_keys(test_dataset, standardized_parameters)

# Save datasets to model-specific folder
cld.save_datasets(train_dataset_std, val_dataset_std, test_dataset_std, files_names)

# Save standardized parameters to reusable folder (only if not already there)
if not os.path.exists(files_names['std_parameters']):
    cld.save_json(standardized_parameters, files_names['std_parameters'])

# Defining target factor
target_factor = np.array(standardized_parameters['target_std']) / standardized_parameters['scale']
target_mean    = standardized_parameters['target_mean']

Define data loaders.

In [ ]:
train_loader = DataLoader(train_dataset_std, batch_size=batch_size, shuffle=False, pin_memory=True)
val_loader   = DataLoader(val_dataset_std,   batch_size=batch_size, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_dataset_std,  batch_size=batch_size, shuffle=False, pin_memory=True)

# Determine number of node-level features in dataset, considering the t_step information
n_node_features = train_dataset_std[0].num_node_features

# Generate Graph Neural Network model

In [ ]:
model = clm.load_model(
    model_type=model_type,
    n_node_features=n_node_features,
    pdropout=dropout,
    device=device,
    model_name=files_names['model'],
    mode='train',
    n_outputs=n_outputs
)

# Train

Define training optimized and criterion

In [ ]:
# MSELoss is by default defined as the mean within the batch
optimizer_dict = {
    "Adam": torch.optim.Adam(model.parameters(), lr=learning_rate),
    "AdamW": torch.optim.AdamW(model.parameters(), lr=learning_rate),
    "RAdam": torch.optim.RAdam(model.parameters(), lr=learning_rate),
    "SGD": torch.optim.SGD(model.parameters(), lr=learning_rate),
    "RMSprop": torch.optim.RMSprop(model.parameters(), lr=learning_rate),
}

loss_function_dict = {
    "MSELoss": torch.nn.MSELoss(),
    "L1Loss": torch.nn.L1Loss(),
    "SmoothL1Loss": torch.nn.SmoothL1Loss(),
}

optimizer = optimizer_dict[optimizer]
criterion = loss_function_dict[loss_function]

# Initialize early stopping
early_stopping = clm.EarlyStopping(patience=patience, delta=delta, model_name=files_names['model'])

In [ ]:
# Train the model
train_losses = []
val_losses   = []
for epoch in np.arange(0, n_epochs):
    train_loss, train_predictions, train_ground_truths = clm.train(model, criterion, train_loader,
                                                                   target_factor,
                                                                   standardized_parameters['target_mean'],
                                                                   optimizer)
    val_loss,   val_predictions,   val_ground_truths   =  clm.test(model, criterion, val_loader,
                                                                   target_factor,
                                                                   standardized_parameters['target_mean'])

    # Convert to original units
    train_loss = np.sum(np.sqrt(train_loss) * target_factor)
    val_loss   = np.sum(np.sqrt(val_loss)   * target_factor)

    if epoch % 5 == 0:
        # Un plot por target (en modo 3d sólo hay uno)
        trn_p  = np.array(train_predictions).reshape(-1, n_outputs)
        trn_t  = np.array(train_ground_truths).reshape(-1, n_outputs)
        val_p  = np.array(val_predictions).reshape(-1, n_outputs)
        val_t  = np.array(val_ground_truths).reshape(-1, n_outputs)
        for t_idx, t_name in enumerate(mis_targets):
            epoch_plot_path = f'{timestamped_folder}/plot_epoch_{epoch}_{t_name}.png'
            cld.parity_plot(
                train=np.array([trn_t[:, t_idx], trn_p[:, t_idx]]),
                validation=np.array([val_t[:, t_idx], val_p[:, t_idx]]),
                save_to=epoch_plot_path,
                title=f'Epoch {epoch} - {t_name}'
            )
            print(f'Parity plot saved to: {epoch_plot_path}')

    # Append losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Check early stopping criteria
    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print('Early stopping')
        break

    print(f'Epoch: {epoch+1}, Train MAE: {train_loss:.4f}, Val MAE: {val_loss:.4f}')

In [ ]:
cld.losses_plot(train_losses=train_losses,
                val_losses=val_losses,
                to_log=True)

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Generar una animación por target
for t_name in mis_targets:
    plot_pattern = f'{timestamped_folder}/plot_epoch_*_{t_name}.png'
    plot_files = sorted(glob.glob(plot_pattern))
    if not plot_files:
        print(f'No se encontraron plots de epoch para {t_name} en {timestamped_folder}')
        continue

    images = [mpimg.imread(path) for path in plot_files]
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(images[0])
    ax.axis('off')
    title = ax.text(0.5, 1.05, '', ha='center', va='bottom', transform=ax.transAxes, fontsize=14)

    def make_update(plot_files_local, t_name_local, title_local, im_local):
        def update(frame):
            im_local.set_array(images[frame])
            parts = os.path.basename(plot_files_local[frame]).split('_')
            # filename: plot_epoch_{epoch}_{t_name}.png
            epoch_num = parts[2]
            title_local.set_text(f'Epoch {epoch_num} - {t_name_local}')
            return im_local, title_local
        return update

    anim = FuncAnimation(fig, make_update(plot_files, t_name, title, im),
                         frames=len(images), interval=800, blit=True)
    animation_path = f'{timestamped_folder}/training_epochs_animation_{t_name}.gif'
    try:
        anim.save(animation_path, writer='pillow', dpi=80)
        print(f'Animation saved to: {animation_path}')
    except Exception as e:
        print('No se pudo guardar la animación GIF:', e)

    display(HTML(anim.to_jshtml()))

# Check test data

In [ ]:
model = clm.load_model(
    model_type=model_type,
    n_node_features=n_node_features,
    pdropout=dropout,
    device=device,
    model_name=files_names['model'],
    mode='eval',
    n_outputs=n_outputs
)


In [ ]:
train_loss, train_predictions, train_ground_truths = clm.test(model, criterion, train_loader,
                                                              target_factor,
                                                              standardized_parameters['target_mean'])
val_loss,   val_predictions,   val_ground_truths   = clm.test(model, criterion, val_loader,
                                                              target_factor,
                                                              standardized_parameters['target_mean'])
test_loss,  test_predictions,  test_ground_truths  = clm.test(model, criterion, test_loader,
                                                              target_factor,
                                                              standardized_parameters['target_mean'])

# Pass to energy units
train_loss = np.sum(np.sqrt(train_loss) * target_factor)
val_loss   = np.sum(np.sqrt(val_loss)   * target_factor)
test_loss  = np.sum(np.sqrt(test_loss)  * target_factor)

# Parity plots: uno por target
trn_p  = np.array(train_predictions).reshape(-1, n_outputs)
trn_t  = np.array(train_ground_truths).reshape(-1, n_outputs)
val_p  = np.array(val_predictions).reshape(-1, n_outputs)
val_t  = np.array(val_ground_truths).reshape(-1, n_outputs)
tst_p  = np.array(test_predictions).reshape(-1, n_outputs)
tst_t  = np.array(test_ground_truths).reshape(-1, n_outputs)
for t_idx, t_name in enumerate(mis_targets):
    cld.parity_plot(
        train=np.array([trn_t[:, t_idx], trn_p[:, t_idx]]),
        validation=np.array([val_t[:, t_idx], val_p[:, t_idx]]),
        test=np.array([tst_t[:, t_idx], tst_p[:, t_idx]]),
        save_to=f'{timestamped_folder}/model-training-{t_name}.pdf',
        title=t_name
    )

print(f'Train MAE: {train_loss:.4f}, Val MAE: {val_loss:.4f}, Test MAE: {test_loss:.4f}')

In [ ]:
import json

def to_serializable(arr):
    """Convierte array 1D o 2D a lista serializable (dict por target si es multitarget)."""
    arr = np.array(arr)
    if arr.ndim == 1 or n_outputs == 1:
        return arr.flatten().tolist()
    # multitarget → dict con clave = nombre del target
    arr2d = arr.reshape(-1, n_outputs)
    return {mis_targets[i]: arr2d[:, i].tolist() for i in range(n_outputs)}

# Save detailed results to the timestamped model folder
results_data = {
    'timestamp': timestamp,
    'targets': mis_targets,
    'model_parameters': model_parameters,
    'training_history': {
        'train_losses': [float(loss) for loss in train_losses],
        'val_losses': [float(loss) for loss in val_losses],
        'epochs_completed': len(train_losses)
    },
    'final_metrics': {
        'train_mae': float(train_loss),
        'val_mae': float(val_loss),
        'test_mae': float(test_loss)
    },
    'predictions': {
        'train': {
            'predicted': to_serializable(train_predictions),
            'actual':    to_serializable(train_ground_truths)
        },
        'validation': {
            'predicted': to_serializable(val_predictions),
            'actual':    to_serializable(val_ground_truths)
        },
        'test': {
            'predicted': to_serializable(test_predictions),
            'actual':    to_serializable(test_ground_truths)
        }
    }
}

results_file = f'{timestamped_folder}/model_results_{timestamp}.json'
with open(results_file, 'w') as f:
    json.dump(results_data, f, indent=4)

print(f'Model results saved to: {results_file}')

In [ ]:
import matplotlib.pyplot as plt

results_file = f'{timestamped_folder}/model_results_{timestamp}.json'
print(f"Cargando resultados desde: {results_file}")

with open(results_file, 'r') as f:
    results_data = json.load(f)

# Extraer datos comunes
train_losses  = results_data['training_history']['train_losses']
val_losses    = results_data['training_history']['val_losses']
saved_targets = results_data.get('targets', ['E_3D'])
n_saved       = len(saved_targets)

# ==========================================
# 1. Curva de Aprendizaje
# ==========================================
print("\n=== Curva de Aprendizaje (Loss) ===")
plt.figure(figsize=(6, 4))
cld.losses_plot(train_losses, val_losses, to_log=True)
loss_plot_path = f'{timestamped_folder}/training_loss_log.png'
plt.savefig(loss_plot_path, dpi=300, bbox_inches='tight')
print(f'Loss chart saved to: {loss_plot_path}')
plt.show()

# ==========================================
# 2. Predicción vs Realidad por target
# ==========================================
def extract_target(data_dict, t_name, n_t):
    """Extrae array de predicciones para un target (soporta 1D y multitarget)."""
    if n_t == 1:
        return np.array(data_dict)
    return np.array(data_dict[t_name])

for t_name in saved_targets:
    predicted = extract_target(results_data['predictions']['test']['predicted'], t_name, n_saved)
    actual    = extract_target(results_data['predictions']['test']['actual'],    t_name, n_saved)

    print(f"\n=== Predicción vs Realidad — {t_name} (Test) ===")
    plt.figure(figsize=(6, 6))
    plt.scatter(actual, predicted, alpha=0.6, edgecolors='w', label='Predicciones')
    min_val = min(np.min(actual), np.min(predicted))
    max_val = max(np.max(actual), np.max(predicted))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Predicción Perfecta (y=x)')
    plt.xlabel(f'{t_name} Calculada / Real')
    plt.ylabel(f'{t_name} Predicha por la GNN')
    plt.title(f'Rendimiento del Modelo en Test — {t_name}')
    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.7)
    pred_plot_path = f'{timestamped_folder}/prediction_vs_real_{t_name}.png'
    plt.savefig(pred_plot_path, dpi=300, bbox_inches='tight')
    print(f'Prediction vs Real plot saved to: {pred_plot_path}')
    plt.show()

# ==========================================
# 3. Métricas
# ==========================================
print("\n=== Métricas Finales (MAE) ===")
print(f"Train MAE: {results_data['final_metrics']['train_mae']:.4f}")
print(f"Val MAE:   {results_data['final_metrics']['val_mae']:.4f}")
print(f"Test MAE:  {results_data['final_metrics']['test_mae']:.4f}")

In [ ]:
import importlib
importlib.reload(cld)